# Fuel Lattice Parameter: Crystal System and Threshold Optimization Study

Sweeps the Lumped-RF-style model (`RandomForestRegressor(n_estimators=600,
random_state=42)`, the same estimator as the shipped `rf1`, but held out on a
train/test split rather than trained on the full data) across every crystal system and
two lattice-parameter thresholds (10 Å, 20 Å), to characterize where the model performs
best.

The estimator and evaluation logic live in `engine/optimize.py`'s `evaluate_model()`,
this notebook only supplies the sweep grid. Output is written to
`Results/CrystalSystemRandomForestRegressorOptimizationStudy_Final.csv` (the canonical
file; see `Results/README.md` for the provenance of the other, non-canonical CSVs in
`Results/archive/`).

**Note on dedup keys:** this study intentionally dedups on
`("composition_reduced", "spacegroup_num")`, i.e. `engine.optimize.DEDUP_KEYS_STUDY`,
which is *not* the same dedup key the training pipeline uses
(`engine.optimize.DEDUP_KEYS_TRAIN`, which also includes `nsites`). This reproduces the
canonical CSV exactly; see `Results/README.md` before changing it.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import joblib
import pandas as pd

from engine import config
from engine.optimize import evaluate_model, DEDUP_KEYS_STUDY

## Load the featurized dataset and feature labels

In [ ]:
df = pd.read_csv(config.DATASET_FEATURIZED, low_memory=False)
display(df.head())

feature_labels = joblib.load(config.FEATURELABELS)
cs_feature_labels = joblib.load(config.FEATURE_DIR / "cs_FeatureLabels.joblib")
sg_feature_labels = joblib.load(config.FEATURE_DIR / "sg_FeatureLabels.joblib")

## Run the sweep

Evaluates every combination of lattice-parameter threshold (10 Å, 20 Å) and crystal
system, using the space-group one-hot encoding (`use_space_group_ohe=True`), matching
the original study.

In [ ]:
CRYSTAL_SYSTEMS = [
    "cubic", "hexagonal", "tetragonal", "orthorhombic",
    "monoclinic", "triclinic", "trigonal",
]

results = []
for lat_thresh in [10, 20]:
    for crys in CRYSTAL_SYSTEMS:
        result = evaluate_model(
            df, feature_labels, sg_feature_labels, cs_feature_labels,
            lat_param_thresh=lat_thresh,
            use_space_group_ohe=True,
            crystal_system=crys,
            dedup_keys=DEDUP_KEYS_STUDY,
        )
        results.append(result)
        print(f"threshold={lat_thresh} Å, crystal_system={crys}: "
              f"dataset_size={result['dataset_size']}, R2={result['R2']}")

df_results = pd.DataFrame(results)
display(df_results)

## Save the canonical study CSV

In [ ]:
out_path = config.REPO_ROOT / "Results" / "CrystalSystemRandomForestRegressorOptimizationStudy_Final.csv"
df_results.to_csv(out_path, index=False)
print(f"Saved: {out_path}")